# WiCount CSI Final Dataset Validation

This notebook performs a final, non-destructive validation of the 15 WiCount CSV recordings and the existing 50-sample feature matrix. Timestamp and sequence differences are computed in original row order (`Series.diff()`); no rows are dropped. Feature reduction is a recommendation only: the baseline feature matrix remains read-only. No model is trained.

In [1]:
import hashlib
import re
import zipfile
from collections import Counter
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

root = Path(r'G:\My Drive\github\projects\WiFiSANSE')
archive_path = root / 'analysis' / 'dataset' / 'WiCount' / 'WiCount.zip'
feature_path = root / 'analysis' / 'processed' / 'features' / 'feature_matrix_50.csv'
prd_path = root / 'PRD.md'
processed_dir = root / 'analysis' / 'processed' / 'features'
output_dir = root / 'analysis' / 'outputs' / 'final_validation'
output_dir.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

protected_hashes_before = {
    'archive': sha256(archive_path),
    'feature_matrix_50': sha256(feature_path),
    'PRD': sha256(prd_path),
}
print('Protected input checksums captured.')
print('Final validation output:', output_dir)


Protected input checksums captured.
Final validation output: G:\My Drive\github\projects\WiFiSANSE\analysis\outputs\final_validation


## 1. Timestamp and sequence audit in original packet order

The CSV row order is treated as packet order. The primary timestamp and sequence deltas are calculated directly with `diff()` and are never computed after sorting. A timestamp gap is flagged descriptively when a positive delta exceeds three times the median positive delta within that recording; this is a heuristic, not a dataset-defined packet-loss threshold.

In [2]:
audit_rows = []
interval_rows = []
with zipfile.ZipFile(archive_path) as archive:
    csv_names = sorted(name for name in archive.namelist() if name.lower().endswith('.csv'))
    for name in csv_names:
        frame = pd.read_csv(archive.open(name))
        timestamp = pd.to_datetime(frame['timestamp'], errors='coerce')
        timestamp_diff = timestamp.diff()
        timestamp_diff_ms = timestamp_diff.dt.total_seconds() * 1000.0
        seq_original = frame['seq']
        seq_numeric = pd.to_numeric(seq_original, errors='coerce')
        seq_diff = seq_numeric.diff()

        positive = timestamp_diff_ms[timestamp_diff_ms > 0].dropna()
        positive_median = float(positive.median()) if len(positive) else np.nan
        gap_threshold = 3.0 * positive_median if np.isfinite(positive_median) else np.nan
        unusual_gap = int((positive > gap_threshold).sum()) if np.isfinite(gap_threshold) else 0
        recording_id = Path(name).stem
        occupancy = int(name.split('/')[1])

        audit_rows.append({
            'recording_id': recording_id,
            'occupancy': occupancy,
            'rows_read': int(len(frame)),
            'timestamp_missing': int(frame['timestamp'].isna().sum()),
            'timestamp_parse_fail': int((frame['timestamp'].notna() & timestamp.isna()).sum()),
            'timestamp_duplicate_rows': int(timestamp.duplicated().sum()),
            'timestamp_diff_zero': int((timestamp_diff == pd.Timedelta(0)).sum()),
            'timestamp_diff_negative': int((timestamp_diff < pd.Timedelta(0)).sum()),
            'timestamp_diff_positive': int((timestamp_diff > pd.Timedelta(0)).sum()),
            'timestamp_diff_missing_first_or_invalid': int(timestamp_diff.isna().sum()),
            'positive_timestamp_diff_median_ms': positive_median,
            'gap_threshold_3x_median_ms': gap_threshold,
            'timestamp_gap_gt_3x_median_count': unusual_gap,
            'timestamp_max_positive_diff_ms': float(positive.max()) if len(positive) else np.nan,
            'seq_missing': int(seq_original.isna().sum()),
            'seq_parse_fail': int((seq_original.notna() & seq_numeric.isna()).sum()),
            'seq_duplicate_rows': int(seq_numeric.duplicated().sum()),
            'seq_diff_zero': int((seq_diff == 0).sum()),
            'seq_diff_negative': int((seq_diff < 0).sum()),
            'seq_diff_positive': int((seq_diff > 0).sum()),
            'seq_diff_nonunit_nonzero': int((seq_diff.notna() & (seq_diff != 0) & (seq_diff != 1)).sum()),
            'seq_diff_missing_first_or_invalid': int(seq_diff.isna().sum()),
            'seq_all_diffs_plus_one': bool(seq_diff.dropna().eq(1).all()),
            'timestamp_diff_positive_seq_diff_nonpositive': int(((timestamp_diff_ms > 0) & (seq_diff <= 0)).sum()),
            'timestamp_diff_nonpositive_seq_diff_positive': int(((timestamp_diff_ms <= 0) & (seq_diff > 0)).sum()),
        })
        local = pd.DataFrame({
            'recording_id': recording_id,
            'occupancy': occupancy,
            'timestamp_diff_ms': timestamp_diff_ms,
            'seq_diff': seq_diff,
        })
        interval_rows.append(local)

timestamp_audit = pd.DataFrame(audit_rows).sort_values(['occupancy', 'recording_id']).reset_index(drop=True)
timestamp_audit.to_csv(processed_dir / 'recording_timestamp_audit.csv', index=False)
intervals = pd.concat(interval_rows, ignore_index=True)
print('Recordings:', len(timestamp_audit))
print('Rows read:', int(timestamp_audit['rows_read'].sum()))
print('Timestamp duplicates:', int(timestamp_audit['timestamp_duplicate_rows'].sum()))
print('Timestamp diff zero / negative / positive:', int(timestamp_audit['timestamp_diff_zero'].sum()), int(timestamp_audit['timestamp_diff_negative'].sum()), int(timestamp_audit['timestamp_diff_positive'].sum()))
print('Timestamp missing / parse failures:', int(timestamp_audit['timestamp_missing'].sum()), int(timestamp_audit['timestamp_parse_fail'].sum()))
print('Sequence duplicates / non-unit nonzero deltas:', int(timestamp_audit['seq_duplicate_rows'].sum()), int(timestamp_audit['seq_diff_nonunit_nonzero'].sum()))
print('Sequence missing / parse failures:', int(timestamp_audit['seq_missing'].sum()), int(timestamp_audit['seq_parse_fail'].sum()))
print('Unusual timestamp gaps (>3x within-recording positive median):', int(timestamp_audit['timestamp_gap_gt_3x_median_count'].sum()))
print(timestamp_audit[['recording_id','occupancy','rows_read','timestamp_duplicate_rows','timestamp_diff_zero','timestamp_diff_negative','timestamp_diff_positive','seq_duplicate_rows','seq_diff_nonunit_nonzero','timestamp_gap_gt_3x_median_count']].to_string(index=False))


Recordings: 15
Rows read: 78618
Timestamp duplicates: 8900
Timestamp diff zero / negative / positive: 8900 0 69703
Timestamp missing / parse failures: 0 0
Sequence duplicates / non-unit nonzero deltas: 0 0
Sequence missing / parse failures: 0 0
Unusual timestamp gaps (>3x within-recording positive median): 32
                 recording_id  occupancy  rows_read  timestamp_duplicate_rows  timestamp_diff_zero  timestamp_diff_negative  timestamp_diff_positive  seq_duplicate_rows  seq_diff_nonunit_nonzero  timestamp_gap_gt_3x_median_count
2023-12-21_19-23-53-985_104_1          0       5218                      1341                 1341                        0                     3876                   0                         0                                 0
2023-12-21_19-27-21-930_104_1          1       4780                       845                  845                        0                     3934                   0                         0                                15
20

In [3]:
# Timestamp differences and sequence increments remain aligned by source row order.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
valid_diff = intervals['timestamp_diff_ms'].dropna()
axes[0].hist(valid_diff, bins=70, color='slateblue', alpha=0.85)
axes[0].axvline(0, color='black', linewidth=1)
axes[0].set_title('Timestamp.diff() in original row order')
axes[0].set_xlabel('Timestamp delta (ms)')
axes[0].set_ylabel('Adjacent packet pairs')
axes[1].bar(timestamp_audit['recording_id'], timestamp_audit['timestamp_duplicate_rows'], color='darkorange', label='repeated timestamp rows')
axes[1].bar(timestamp_audit['recording_id'], timestamp_audit['timestamp_gap_gt_3x_median_count'], color='teal', alpha=0.8, label='large positive gaps (>3x median)')
axes[1].set_title('Recording timestamp quality counts')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=70)
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(output_dir / 'timestamp_order_audit.png', dpi=180)
plt.close(fig)

fig, ax = plt.subplots(figsize=(11, 4))
seq_delta_counts = intervals['seq_diff'].value_counts(dropna=True).sort_index()
ax.bar(seq_delta_counts.index.astype(str), seq_delta_counts.values, color='seagreen')
ax.set_title('Sequence-number delta distribution in original row order')
ax.set_xlabel('seq.diff()')
ax.set_ylabel('Adjacent packet pairs')
fig.tight_layout()
fig.savefig(output_dir / 'sequence_delta_distribution.png', dpi=180)
plt.close(fig)
print('Saved timestamp and sequence audit plots.')


Saved timestamp and sequence audit plots.


## 2. Feature redundancy and non-destructive reduction recommendation

The existing feature matrix is read only. Correlation counts use absolute Pearson correlation on the 992 numeric columns. Raw-variance thresholds are descriptive because the feature families have different units and scales; they are not automatic deletion criteria. The compact feature set is a candidate for later train-only ablation, not a proven optimal set.

In [4]:
feature_frame = pd.read_csv(feature_path)
metadata_cols = ['window_id', 'recording_id', 'occupancy', 'window_size', 'start_index', 'end_index', 'row_count']
missing_meta = sorted(set(metadata_cols) - set(feature_frame.columns))
if missing_meta:
    raise ValueError(f'Missing expected metadata columns: {missing_meta}')
numeric_cols = [c for c in feature_frame.columns if c not in metadata_cols]
features = feature_frame[numeric_cols].apply(pd.to_numeric, errors='coerce')
if features.isna().any().any() or np.isinf(features.to_numpy(dtype=float)).any():
    raise ValueError('Feature matrix contains NaN or infinite numeric values.')

variance = features.var(axis=0, ddof=1)
constant_features = variance[variance <= 1e-12].index.tolist()
low_variance_features = variance[variance < 0.01].sort_values()
cross_std_var_max_abs_error = float((feature_frame['cross_subcarrier_var_amp'] - feature_frame['cross_subcarrier_std_amp'] ** 2).abs().max())
if not np.isclose(cross_std_var_max_abs_error, 0.0, atol=1e-12):
    raise ValueError(f'Cross-subcarrier variance/std relation failed: {cross_std_var_max_abs_error}')
correlation = features.corr(method='pearson').abs()
upper = correlation.where(np.triu(np.ones(correlation.shape, dtype=bool), k=1))
high_pairs = upper.stack().reset_index()
high_pairs.columns = ['feature_a', 'feature_b', 'abs_pearson_r']
high_pairs = high_pairs[high_pairs['abs_pearson_r'] >= 0.90].sort_values('abs_pearson_r', ascending=False).reset_index(drop=True)

def feature_parts(name):
    match = re.fullmatch(r'sub_(\d+)_(.+)', name)
    if match:
        return 'per_subcarrier', int(match.group(1)), match.group(2)
    if name.startswith('cross_subcarrier_'):
        return 'cross_subcarrier', None, name[len('cross_subcarrier_'):]
    return 'other', None, name

high_pairs[['scope_a','subcarrier_a','type_a']] = pd.DataFrame(high_pairs['feature_a'].map(feature_parts).tolist(), index=high_pairs.index)
high_pairs[['scope_b','subcarrier_b','type_b']] = pd.DataFrame(high_pairs['feature_b'].map(feature_parts).tolist(), index=high_pairs.index)
high_pairs['type_pair'] = high_pairs.apply(lambda r: ' vs '.join(sorted([r['type_a'], r['type_b']])), axis=1)
high_pairs['relation'] = high_pairs.apply(lambda r: 'same_subcarrier' if pd.notna(r['subcarrier_a']) and r['subcarrier_a'] == r['subcarrier_b'] else ('cross_subcarrier' if pd.notna(r['subcarrier_a']) and pd.notna(r['subcarrier_b']) else 'includes_cross_feature'), axis=1)
high_pairs.to_csv(output_dir / 'high_correlation_pair_details.csv', index=False)

# Exact duplicate numeric columns, confirmed by value equality after hash grouping.
hash_groups = {}
for column in numeric_cols:
    key = pd.util.hash_pandas_object(features[column], index=False).values.tobytes()
    hash_groups.setdefault(key, []).append(column)
exact_duplicate_pairs = []
for columns in hash_groups.values():
    if len(columns) < 2:
        continue
    for i, first in enumerate(columns):
        for second in columns[i + 1:]:
            if np.array_equal(features[first].to_numpy(), features[second].to_numpy()):
                exact_duplicate_pairs.append((first, second))
exact_partner = {}
for first, second in exact_duplicate_pairs:
    exact_partner.setdefault(first, []).append(second)
    exact_partner.setdefault(second, []).append(first)

core_types = {'mean', 'std', 'zero_crossing_rate', 'first_diff_std'}
recommendation_rows = []
for column in numeric_cols:
    scope, subcarrier, feature_type = feature_parts(column)
    max_corr = float(correlation[column].drop(labels=[column]).max()) if len(numeric_cols) > 1 else np.nan
    high_degree = int((correlation[column].drop(labels=[column]) >= 0.90).sum())
    if column in exact_partner and feature_type == 'temporal_std':
        tier = 'exclude_exact_duplicate_from_initial_compact_set'
        rationale = 'Exact duplicate of the corresponding std feature; keep std in the candidate core and preserve both columns in the baseline.'
    elif column == 'cross_subcarrier_var_amp':
        tier = 'exclude_algebraically_redundant_from_initial_compact_set'
        rationale = 'Cross-subcarrier variance is the square of cross-subcarrier standard deviation under the same ddof; retain baseline, test only if useful.'
    elif (scope == 'per_subcarrier' and feature_type in core_types) or column in {'cross_subcarrier_mean_amp','cross_subcarrier_std_amp','cross_subcarrier_corr_median'}:
        tier = 'candidate_core'
        rationale = 'Parsimonious candidate family for later session-grouped, train-only feature ablation; not validated for predictive utility.'
    else:
        tier = 'optional_ablation_not_in_initial_compact_set'
        rationale = 'Retain in baseline; compare on training folds only if a later experiment justifies the added feature family.'
    recommendation_rows.append({
        'feature': column,
        'scope': scope,
        'subcarrier': subcarrier,
        'feature_type': feature_type,
        'sample_variance': float(variance[column]),
        'variance_below_0_01_raw_units': bool(variance[column] < 0.01),
        'exact_duplicate_partners': '|'.join(exact_partner.get(column, [])),
        'max_abs_pearson_correlation': max_corr,
        'count_abs_pearson_ge_0_90': high_degree,
        'recommendation_tier': tier,
        'rationale': rationale,
    })
recommendations = pd.DataFrame(recommendation_rows)
recommendations.to_csv(processed_dir / 'feature_reduction_recommendation.csv', index=False)

print('Feature matrix shape:', feature_frame.shape)
print('Numeric feature count:', len(numeric_cols))
print('Metadata kept separate:', metadata_cols)
print('High-correlation pairs (|r| >= 0.90):', len(high_pairs))
print('Same-subcarrier pairs:', int((high_pairs['relation'] == 'same_subcarrier').sum()))
print('Cross-subcarrier pairs:', int((high_pairs['relation'] == 'cross_subcarrier').sum()))
print('Pairs including cross-subcarrier aggregate feature:', int((high_pairs['relation'] == 'includes_cross_feature').sum()))
print('Exact duplicate numeric-column pairs:', len(exact_duplicate_pairs))
print('Constant features (variance <= 1e-12):', len(constant_features))
print('Low raw-variance features (variance < 0.01):', len(low_variance_features))
print('Cross-subcarrier variance vs std squared max absolute error:', cross_std_var_max_abs_error)
print('Candidate compact feature count:', int((recommendations['recommendation_tier'] == 'candidate_core').sum()))
print('Most frequent feature-type pairs:')
print(high_pairs['type_pair'].value_counts().head(15).to_string())
print('Lowest-variance features:')
print(variance.sort_values().head(10).to_string())


Feature matrix shape: (3122, 999)
Numeric feature count: 992
Metadata kept separate: ['window_id', 'recording_id', 'occupancy', 'window_size', 'start_index', 'end_index', 'row_count']
High-correlation pairs (|r| >= 0.90): 3892
Same-subcarrier pairs: 1362
Cross-subcarrier pairs: 2529
Pairs including cross-subcarrier aggregate feature: 1
Exact duplicate numeric-column pairs: 52
Constant features (variance <= 1e-12): 0
Low raw-variance features (variance < 0.01): 52
Cross-subcarrier variance vs std squared max absolute error: 1.4210854715202004e-14
Candidate compact feature count: 211
Most frequent feature-type pairs:
type_pair
std vs temporal_std    154
mad vs temporal_std    147
mad vs std             147
mean vs rms            144
mean vs median         141
energy vs rms          140
energy vs mean         139
median vs rms          138
std vs var             137
temporal_std vs var    137
energy vs median       134
mad vs var             132
q3 vs rms              125
energy vs q3    

In [5]:
# Visual summaries for feature redundancy and variance.
feature_pair_counts = high_pairs['type_pair'].value_counts().head(15).sort_values()
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(feature_pair_counts.index, feature_pair_counts.values, color='firebrick')
ax.set_title('Most common high-correlation feature-type pairs')
ax.set_xlabel('Pair count (|Pearson r| >= 0.90)')
fig.tight_layout()
fig.savefig(output_dir / 'feature_redundancy_by_type.png', dpi=180)
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5))
ordered_variance = variance.sort_values().to_numpy()
ax.plot(np.arange(1, len(ordered_variance) + 1), ordered_variance, color='steelblue')
ax.axhline(0.01, color='darkorange', linestyle='--', label='descriptive threshold 0.01')
ax.set_yscale('log')
ax.set_title('Sorted numeric feature variances (raw feature scales)')
ax.set_xlabel('Feature rank by variance')
ax.set_ylabel('Sample variance (log scale)')
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / 'feature_variance_rank.png', dpi=180)
plt.close(fig)
print('Saved feature redundancy and variance plots.')


Saved feature redundancy and variance plots.


## 3. Group-based validation feasibility

All windows from a recording are one indivisible group. For classes with at least two recordings, the table illustrates capacity for one recording in a held-out validation/test role and the remaining recordings in training. For a singleton class, the only recording can be in training but there is no independent same-class validation/test recording; holding it out would instead leave the training set without that class. This is a feasibility illustration, not an actual split.

In [6]:
window_counts = feature_frame.groupby(['occupancy', 'recording_id'], as_index=False).size().rename(columns={'size': 'window_count'})
recording_counts = timestamp_audit.groupby('occupancy', as_index=False).agg(recording_count=('recording_id','count'))
window_totals = window_counts.groupby('occupancy', as_index=False).agg(window_count=('window_count','sum'))
feasibility = recording_counts.merge(window_totals, on='occupancy', how='outer').fillna(0)
feasibility['recordings_in_training_if_one_group_held_out'] = feasibility['recording_count'].apply(lambda n: int(n - 1) if n >= 2 else int(n))
feasibility['recordings_in_validation_or_test_if_one_group_held_out'] = feasibility['recording_count'].apply(lambda n: 1 if n >= 2 else 0)
feasibility['class_specific_independent_evaluation_feasible'] = feasibility['recording_count'] >= 2
feasibility['note'] = feasibility['recording_count'].apply(lambda n: 'Singleton: training-only if class must be represented in train; no independent same-class evaluation.' if n == 1 else 'Can hold out a recording while retaining same-class recordings in train; does not establish broad generalization.')
feasibility = feasibility.rename(columns={'occupancy':'occupancy_class','recording_count':'recording_count','window_count':'window_count'})
feasibility.to_csv(processed_dir / 'group_split_feasibility.csv', index=False)
window_counts.to_csv(output_dir / 'window_counts_by_recording.csv', index=False)
print(feasibility.to_string(index=False))
print('Total windows:', int(feasibility['window_count'].sum()))

fig, ax = plt.subplots(figsize=(8, 5))
classes = feasibility['occupancy_class'].astype(str)
positions = np.arange(len(classes))
ax.bar(positions - 0.18, feasibility['recording_count'], width=0.36, label='recordings', color='slateblue')
ax.bar(positions + 0.18, feasibility['recordings_in_validation_or_test_if_one_group_held_out'], width=0.36, label='one-group holdout capacity', color='darkorange')
ax.set_xticks(positions)
ax.set_xticklabels(classes)
ax.set_title('Recording-group split feasibility by occupancy')
ax.set_xlabel('Occupancy class')
ax.set_ylabel('Recording count')
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / 'group_split_feasibility.png', dpi=180)
plt.close(fig)
print('Saved group split feasibility plot.')


 occupancy_class  recording_count  window_count  recordings_in_training_if_one_group_held_out  recordings_in_validation_or_test_if_one_group_held_out  class_specific_independent_evaluation_feasible                                                                                                              note
               0                1           207                                             1                                                       0                                           False             Singleton: training-only if class must be represented in train; no independent same-class evaluation.
               1                7          1452                                             6                                                       1                                            True Can hold out a recording while retaining same-class recordings in train; does not establish broad generalization.
               2                6          1268                 

Saved group split feasibility plot.


## 4. Preprocessing leakage audit

Metadata are excluded explicitly from the numeric feature matrix. The existing standardized PCA used all windows and is descriptive only; it is not a leakage-free model preprocessing pipeline. The earlier DSP PCA excluded metadata but centered unscaled columns and is also descriptive. The normalization experiment is a descriptive per-recording/per-window comparison, not a fitted model transformer. Any scaler, PCA, correlation-based feature selection, imputation rule, or other data-learned preprocessing used in future ML must be fitted on training recordings only inside each validation fold. No random-window split is permitted.

In [7]:
metadata_excluded = set(metadata_cols).isdisjoint(set(numeric_cols))
recording_ids_match = set(feature_frame['recording_id'].unique()) == set(timestamp_audit['recording_id'].unique())
windows_by_recording = feature_frame.groupby('recording_id').size()
labels_per_recording = feature_frame.groupby('recording_id')['occupancy'].nunique()
assert metadata_excluded
assert recording_ids_match
assert labels_per_recording.eq(1).all()
assert len(timestamp_audit) == 15
assert int(timestamp_audit['rows_read'].sum()) == 78618
assert int(feature_frame.shape[0]) == 3122
assert len(numeric_cols) == 992
assert int(timestamp_audit['timestamp_duplicate_rows'].sum()) == 8900
assert int(timestamp_audit['timestamp_diff_zero'].sum()) == 8900
assert int(timestamp_audit['timestamp_diff_negative'].sum()) == 0
assert int(timestamp_audit['timestamp_diff_positive'].sum()) == 69703
assert int(timestamp_audit['seq_duplicate_rows'].sum()) == 0
assert int(timestamp_audit['seq_diff_nonunit_nonzero'].sum()) == 0
assert timestamp_audit['seq_all_diffs_plus_one'].all()
assert int(window_counts['window_count'].sum()) == 3122

protected_hashes_after = {
    'archive': sha256(archive_path),
    'feature_matrix_50': sha256(feature_path),
    'PRD': sha256(prd_path),
}
assert protected_hashes_before == protected_hashes_after, 'A protected input changed during this notebook run.'
print('Metadata/labels excluded from numeric features:', metadata_excluded)
print('Feature recording IDs match all raw recording IDs:', recording_ids_match)
print('Rows dropped by this audit: 0 (read-only audit)')
print('Protected archive, feature matrix, and PRD unchanged:', protected_hashes_before == protected_hashes_after)
print('SHA-256 checksums:', protected_hashes_after)
print('No model training was performed.')


Metadata/labels excluded from numeric features: True
Feature recording IDs match all raw recording IDs: True
Rows dropped by this audit: 0 (read-only audit)
Protected archive, feature matrix, and PRD unchanged: True
SHA-256 checksums: {'archive': 'd578ba4ec4d64ef7089e0fa5c8b47498a76fce5f7d32fb7c7f7f994235915ae1', 'feature_matrix_50': '1471b768656afe8bd5b4b3db8cd9c2c2c012e33c3287963a39c0f540703a141c', 'PRD': 'b97e85a201b7e45b0b11e8a700d4a728be7720ac186844c1bf7df023e18f2fad'}
No model training was performed.
